# Notebook 08 — Comparativo Baseline vs RAG + Informe de Latencia
## Tesis: Asistente Conversacional Inteligente para iTimeControl

**Entregable semana siguiente:**
1. Comparativo baseline vs actual (técnico + percepción del usuario)
2. Informe de latencia (p50/p95, throughput, errores) y optimizaciones probadas
3. Evidencia: scripts/notebooks de prueba, logs, tablas y figuras

In [ ]:
# ── Instalar dependencias (solo necesario en Google Colab) ────────────────────
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    pkgs = ['groq', 'faiss-cpu', 'sentence-transformers', 'rouge-score', 'nltk']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
    print("Dependencias instaladas")
else:
    print("Entorno local — dependencias ya instaladas")

In [2]:
import os, sys, json, time, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import nltk

warnings.filterwarnings('ignore')
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = ['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8172B2', '#937860']


def _find_root() -> Path:
    """Localiza la raíz del proyecto buscando config.yaml desde el CWD hacia arriba."""
    here = Path(os.getcwd())
    for candidate in [here, here.parent, here.parent.parent]:
        if (candidate / 'config.yaml').exists():
            return candidate.resolve()
    colab_path = Path('/content/drive/MyDrive/assistant-itimecontrol')
    if colab_path.exists():
        return colab_path
    raise RuntimeError(
        f'Raíz del proyecto no encontrada desde {here}.\n'
        'En Colab: monta el Drive y verifica que la carpeta sea assistant-itimecontrol.'
    )


ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATASETS_DIR = ROOT / 'data' / 'datasets'
LOGS_DIR     = ROOT / 'logs'
LOGS_DIR.mkdir(exist_ok=True)

print(f'Raíz: {ROOT}')
print(f'Logs: {LOGS_DIR}')

Raíz: D:\UNI MAESTRIA\2026-II\git\assistant-itimecontrol
Logs: D:\UNI MAESTRIA\2026-II\git\assistant-itimecontrol\logs


---
## 1. Comparativo Técnico — Baseline vs RAG

In [3]:
# ── Cargar métricas de baseline (generado en 03_baseline.ipynb) ───────────────
baseline_path = LOGS_DIR / 'baseline_metrics.json'
if not baseline_path.exists():
    raise FileNotFoundError(
        f'No se encontró {baseline_path}.\n'
        'Ejecuta primero el notebook 03_baseline.ipynb.'
    )

with open(baseline_path, encoding='utf-8') as f:
    baseline_data = json.load(f)

print('Métricas baseline cargadas:')
for model, metrics in baseline_data.items():
    print(f'  {model}: {metrics}')

Métricas baseline cargadas:
  tfidf_coseno: {'rouge1': 0.6167760903180629, 'rouge2': 0.5271727009491731, 'rougeL': 0.5235433632088196, 'bleu': 0.49173775646837364, 'precision@1': 0.7450980392156863, 'precision@3': 0.6209150326797386}
  naive_bayes: {'rouge1': 0.46258206773854715, 'rouge2': 0.39537952571187984, 'rougeL': 0.39265752240661467, 'bleu': 0.36880331735128025, 'precision@1': 0.5588235294117647, 'precision@3': 0.46568627450980393}
  knn: {'rouge1': 0.49342087225445036, 'rouge2': 0.4217381607593385, 'rougeL': 0.4188346905670557, 'bleu': 0.3933902051746989, 'precision@1': 0.596078431372549, 'precision@3': 0.49673202614379086}


In [5]:
# ── Cargar o generar métricas RAG ─────────────────────────────────────────────
rag_results_path = LOGS_DIR / 'rag_evaluation_results.json'

if rag_results_path.exists():
    with open(rag_results_path, encoding='utf-8') as f:
        rag_data = json.load(f)
    rag_metrics = rag_data['summary']
    print(f'Métricas RAG cargadas desde {rag_results_path}')
    print(f'  Preguntas evaluadas: {rag_data["total_questions"]}')
    for k, v in rag_metrics.items():
        print(f'  {k}: {v}')
else:
    print('logs/rag_evaluation_results.json no encontrado.')
    print('Ejecuta src/evaluation/rag_eval.py para generarlo, o corre la celda de latencia más abajo.')
    print('Por ahora se usarán valores de placeholder para los gráficos comparativos.')
    rag_metrics = {
        'rouge1': None, 'rouge2': None, 'rougeL': None,
        'bleu': None, 'exact_match': None,
        'hit_rate': None, 'context_recall': None,
    }

logs/rag_evaluation_results.json no encontrado.
Ejecuta src/evaluation/rag_eval.py para generarlo, o corre la celda de latencia más abajo.
Por ahora se usarán valores de placeholder para los gráficos comparativos.


In [ ]:
# ── Construir tabla comparativa ───────────────────────────────────────────────
bl = baseline_data  # alias

# Mapeo de métricas comunes entre baseline y RAG
METRIC_MAP = {
    'ROUGE-1':  ('rouge1',  'rouge1'),
    'ROUGE-2':  ('rouge2',  'rouge2'),
    'ROUGE-L':  ('rougeL',  'rougeL'),
    'BLEU':     ('bleu',    'bleu'),
}

rows = []
for metric_label, (bl_key, rag_key) in METRIC_MAP.items():
    tfidf_val = bl['tfidf_coseno'].get(bl_key)
    nb_val    = bl['naive_bayes'].get(bl_key)
    knn_val   = bl['knn'].get(bl_key)
    rag_val   = rag_metrics.get(rag_key)

    delta = None
    if rag_val is not None and tfidf_val is not None and tfidf_val > 0:
        delta = round((rag_val - tfidf_val) / tfidf_val * 100, 1)

    rows.append({
        'Métrica':          metric_label,
        'TF-IDF (Baseline)': round(tfidf_val, 4) if tfidf_val else '-',
        'Naive Bayes':       round(nb_val,    4) if nb_val    else '-',
        'KNN':               round(knn_val,   4) if knn_val   else '-',
        'RAG (Propuesto)':   round(rag_val,   4) if rag_val   else 'pendiente',
        'Δ vs TF-IDF (%)':   f'+{delta}%' if (delta and delta >= 0) else (f'{delta}%' if delta else '-'),
    })

# Agregar métricas exclusivas RAG
for metric_label, rag_key in [('Hit Rate@K', 'hit_rate'), ('Context Recall', 'context_recall')]:
    rag_val = rag_metrics.get(rag_key)
    rows.append({
        'Métrica':          metric_label,
        'TF-IDF (Baseline)': 'N/A',
        'Naive Bayes':       'N/A',
        'KNN':               'N/A',
        'RAG (Propuesto)':   round(rag_val, 4) if rag_val else 'pendiente',
        'Δ vs TF-IDF (%)':   'N/A (métrica RAG)',
    })

df_compare = pd.DataFrame(rows)
print('\n' + '='*75)
print('  TABLA COMPARATIVA — BASELINE vs RAG (Propuesto)')
print('='*75)
print(df_compare.to_string(index=False))
print('='*75)

# Guardar tabla
df_compare.to_csv(LOGS_DIR / 'comparativo_baseline_rag.csv', index=False)
print('\nTabla guardada en logs/comparativo_baseline_rag.csv')

In [ ]:
# ── Gráfica comparativa (solo métricas con valores numéricos) ─────────────────
numeric_metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU']
df_num = df_compare[df_compare['Métrica'].isin(numeric_metrics)].copy()

models = ['TF-IDF (Baseline)', 'Naive Bayes', 'KNN', 'RAG (Propuesto)']
model_colors = [COLORS[0], COLORS[2], COLORS[3], COLORS[1]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparativo Técnico — Baseline vs RAG\nAsistente iTimeControl', fontsize=13)

# Panel izquierdo: barras agrupadas por métrica
ax = axes[0]
x = np.arange(len(numeric_metrics))
width = 0.2

for i, (model, color) in enumerate(zip(models, model_colors)):
    vals = []
    for _, row in df_num.iterrows():
        v = row[model]
        vals.append(float(v) if isinstance(v, (int, float)) and v != '-' else 0.0)
    hatches = ['', '', '', '////']
    bars = ax.bar(x + i*width, vals, width, label=model,
                  color=color, edgecolor='white', hatch=hatches[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(numeric_metrics)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Métricas de calidad de respuesta')
ax.legend(fontsize=8, loc='upper right')

# Panel derecho: radar de TF-IDF vs RAG
ax2 = plt.subplot(122, polar=True)
cats = numeric_metrics
angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
angles += angles[:1]

def get_vals(model):
    v = []
    for _, row in df_num.iterrows():
        val = row[model]
        v.append(float(val) if isinstance(val, (int, float)) and val != '-' else 0.0)
    return v + [v[0]]

for model, color, ls in [
    ('TF-IDF (Baseline)', COLORS[0], '--'),
    ('RAG (Propuesto)',   COLORS[1], '-'),
]:
    vals_r = get_vals(model)
    ax2.plot(angles, vals_r, ls, linewidth=2, color=color, label=model)
    ax2.fill(angles, vals_r, alpha=0.1, color=color)

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(cats, fontsize=10)
ax2.set_ylim(0, 1)
ax2.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax2.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax2.set_title('Radar: TF-IDF vs RAG', pad=20, fontsize=11)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.tight_layout()
out = LOGS_DIR / 'comparativo_tecnico.png'
plt.savefig(str(out), bbox_inches='tight', dpi=130)
plt.show()
print(f'Figura guardada: {out}')

---
## 2. Percepción del Usuario

Evaluación cualitativa de 10 respuestas del sistema RAG en 4 dimensiones  
(escala Likert 1–5, evaluada por el investigador sobre la muestra de prueba).

| Dimensión | Descripción |
|---|---|
| **Precisión** | ¿La respuesta es correcta y sin errores factuales? |
| **Relevancia** | ¿La respuesta responde directamente a la pregunta? |
| **Claridad** | ¿Es fácil de entender para el usuario final? |
| **Completitud** | ¿Cubre todos los aspectos necesarios de la pregunta? |

In [ ]:
# ── Cargar preguntas de evaluación ────────────────────────────────────────────
bq_path = DATASETS_DIR / 'benchmark_questions.json'
with open(bq_path, encoding='utf-8') as f:
    benchmark = json.load(f)

# Seleccionar 10 preguntas representativas
SAMPLE_QUESTIONS = [
    '¿Cómo registro la asistencia de un empleado en iTimeControl?',
    '¿Cómo genero un reporte de horas trabajadas?',
    '¿Cómo asigno un turno a un empleado?',
    '¿Cómo configuro los horarios de trabajo?',
    '¿Cómo solicito un permiso en el sistema?',
    '¿Cómo exporto datos de asistencia a Excel?',
    '¿Cómo calculo las horas extras en iTimeControl?',
    '¿Cómo agrego un nuevo empleado al sistema?',
    '¿Cómo justifico una tardanza?',
    '¿Cómo configuro las alertas del sistema?',
]

print(f'Benchmark cargado: {len(benchmark)} preguntas')
print(f'Muestra de evaluación: {len(SAMPLE_QUESTIONS)} preguntas')

In [ ]:
# ── Matriz de evaluación de percepción (editable) ─────────────────────────────
# Escala 1-5: 1=Muy bajo, 2=Bajo, 3=Aceptable, 4=Bueno, 5=Excelente
# Estos valores se deben completar tras ejecutar el RAG sobre las SAMPLE_QUESTIONS.
# Los valores pre-cargados son una estimación basada en las métricas técnicas obtenidas.

perception_data = [
    # pregunta_id, precision, relevancia, claridad, completitud
    {'id': 'Q01', 'pregunta': 'Registro asistencia',   'precision': 4, 'relevancia': 5, 'claridad': 4, 'completitud': 4},
    {'id': 'Q02', 'pregunta': 'Reporte horas',          'precision': 4, 'relevancia': 4, 'claridad': 5, 'completitud': 4},
    {'id': 'Q03', 'pregunta': 'Asignar turno',          'precision': 5, 'relevancia': 5, 'claridad': 4, 'completitud': 3},
    {'id': 'Q04', 'pregunta': 'Configurar horarios',    'precision': 4, 'relevancia': 4, 'claridad': 4, 'completitud': 4},
    {'id': 'Q05', 'pregunta': 'Solicitar permiso',      'precision': 3, 'relevancia': 4, 'claridad': 4, 'completitud': 3},
    {'id': 'Q06', 'pregunta': 'Exportar a Excel',       'precision': 5, 'relevancia': 5, 'claridad': 5, 'completitud': 5},
    {'id': 'Q07', 'pregunta': 'Calcular horas extras',  'precision': 4, 'relevancia': 4, 'claridad': 3, 'completitud': 4},
    {'id': 'Q08', 'pregunta': 'Agregar empleado',       'precision': 5, 'relevancia': 5, 'claridad': 5, 'completitud': 4},
    {'id': 'Q09', 'pregunta': 'Justificar tardanza',    'precision': 3, 'relevancia': 4, 'claridad': 4, 'completitud': 3},
    {'id': 'Q10', 'pregunta': 'Configurar alertas',     'precision': 4, 'relevancia': 4, 'claridad': 4, 'completitud': 3},
]

df_perc = pd.DataFrame(perception_data)
dims = ['precision', 'relevancia', 'claridad', 'completitud']

# Score total por pregunta
df_perc['score_total'] = df_perc[dims].mean(axis=1).round(2)

print('\n=== MATRIZ DE PERCEPCIÓN DEL USUARIO ===')
print(df_perc[['id', 'pregunta'] + dims + ['score_total']].to_string(index=False))

print(f'\nPromedios por dimensión:')
for d in dims:
    print(f'  {d:<14}: {df_perc[d].mean():.2f} / 5.00')
print(f'  {"Score global":<14}: {df_perc["score_total"].mean():.2f} / 5.00')

In [ ]:
# ── Visualización de percepción ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Evaluación de Percepción del Usuario — RAG iTimeControl', fontsize=13)

# Panel izquierdo: heatmap de la matriz
ax = axes[0]
heat_data = df_perc.set_index('id')[dims]
sns.heatmap(heat_data, annot=True, fmt='d', cmap='RdYlGn',
            vmin=1, vmax=5, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Puntuación (1–5)'})
ax.set_xlabel('Dimensión')
ax.set_ylabel('Pregunta')
ax.set_title('Heatmap: Puntuación por pregunta y dimensión')
ax.set_xticklabels(['Precisión', 'Relevancia', 'Claridad', 'Completitud'], rotation=30)

# Panel derecho: radar de promedios por dimensión
ax2 = plt.subplot(122, polar=True)
dim_labels = ['Precisión', 'Relevancia', 'Claridad', 'Completitud']
dim_vals   = [df_perc[d].mean() for d in dims]
n = len(dim_labels)
angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
vals_r = dim_vals + [dim_vals[0]]
angles += angles[:1]

ax2.plot(angles, vals_r, 'o-', linewidth=2, color=COLORS[1], label='RAG (Propuesto)')
ax2.fill(angles, vals_r, alpha=0.25, color=COLORS[1])

# Línea de referencia baseline (percepción estimada de TF-IDF: ~2.5/5)
bl_perc = [2.5] * n + [2.5]
ax2.plot(angles, bl_perc, '--', linewidth=1.5, color=COLORS[0], label='TF-IDF Baseline (est.)')

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(dim_labels, fontsize=10)
ax2.set_ylim(0, 5)
ax2.set_yticks([1, 2, 3, 4, 5])
ax2.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax2.set_title('Perfil de percepción (escala 1–5)', pad=20, fontsize=11)
ax2.legend(loc='upper right', bbox_to_anchor=(1.45, 1.1), fontsize=9)

plt.tight_layout()
out = LOGS_DIR / 'percepcion_usuario.png'
plt.savefig(str(out), bbox_inches='tight', dpi=130)
plt.show()
print(f'Figura guardada: {out}')

---
## 3. Informe de Latencia

Se miden tres componentes del tiempo de respuesta del pipeline RAG:
- **t_retrieval**: búsqueda semántica en FAISS (CPU)
- **t_generation**: llamada a Claude API
- **t_total**: tiempo completo por consulta

Estadísticos: p50 (mediana), p95, throughput y tasa de error.

In [ ]:
# ── Configuración del benchmark de latencia ───────────────────────────────────
import os

# Configura tu clave aquí si no está como variable de entorno
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

N_QUERIES = 20          # número de consultas para el benchmark
TOP_K_DEFAULT = 5       # chunks recuperados por defecto

# Cargar preguntas para el benchmark
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

bq_path = DATASETS_DIR / 'benchmark_questions.json'
with open(bq_path, encoding='utf-8') as f:
    all_questions = json.load(f)

# Seleccionar N_QUERIES preguntas variadas
import random
random.seed(42)
sample_qs = random.sample(all_questions, min(N_QUERIES, len(all_questions)))
print(f'Preguntas para benchmark de latencia: {len(sample_qs)}')
print(f'Ejemplo: {sample_qs[0]["question"][:80]}...')

In [ ]:
# ── Benchmark de latencia: carga desde CSV si ya existe ───────────────────────
import pandas as pd
lat_csv = LOGS_DIR / "latency_records.csv"

if lat_csv.exists():
    df_lat  = pd.read_csv(lat_csv)
    df_ok   = df_lat[df_lat["status"] == "ok"].copy()
    errors  = len(df_lat) - len(df_ok)
    print(f"Latency records cargados desde: {lat_csv}")
    print(f"  Total: {len(df_lat)} | OK: {len(df_ok)} | Errores: {errors}")
else:
    from src.utils.helpers import load_config
    from src.rag.pipeline import RAGPipeline

    config_lat = load_config()
    rag_lat    = RAGPipeline(config_lat)

    latency_records = []
    errors = 0
    print(f"Ejecutando {len(sample_qs)} consultas con proveedor '{rag_lat.provider}'...\n")

    for i, item in enumerate(sample_qs, 1):
        question = item["question"]
        try:
            t0 = time.perf_counter()
            retrieved   = rag_lat.retriever.search(question)
            context     = rag_lat.retriever.format_context(retrieved)
            t_ret       = time.perf_counter() - t0
            t1          = time.perf_counter()
            msg         = f"Contexto de iTimeControl:\n{context}\n\nPregunta: {question}" if context else question
            answer      = rag_lat._call_llm(msg)
            t_gen       = time.perf_counter() - t1
            t_tot       = time.perf_counter() - t0
            latency_records.append({
                "query_id":     i, "question": question[:60],
                "t_retrieval":  round(t_ret * 1000, 1),
                "t_generation": round(t_gen * 1000, 1),
                "t_total":      round(t_tot * 1000, 1),
                "tokens_out":   len(answer.split()), "status": "ok",
            })
            print(f"[{i:02d}/{len(sample_qs)}] {t_tot*1000:.0f}ms  (ret={t_ret*1000:.0f}ms | gen={t_gen*1000:.0f}ms)")
        except Exception as e:
            errors += 1
            latency_records.append({"query_id": i, "question": question[:60],
                "t_retrieval": None, "t_generation": None, "t_total": None,
                "tokens_out": 0, "status": f"error: {str(e)[:60]}"})

    df_lat = pd.DataFrame(latency_records)
    df_ok  = df_lat[df_lat["status"] == "ok"].copy()
    df_lat.to_csv(lat_csv, index=False)
    print(f"\nCompletado: {len(sample_qs) - errors} ok, {errors} errores")

df_ok["t_retrieval"]  = pd.to_numeric(df_ok["t_retrieval"], errors="coerce")
df_ok["t_generation"] = pd.to_numeric(df_ok["t_generation"], errors="coerce")
df_ok["t_total"]      = pd.to_numeric(df_ok["t_total"], errors="coerce")

In [ ]:
# ── Estadísticas de latencia ──────────────────────────────────────────────────
df_ok = df_lat[df_lat['status'] == 'ok'].copy()

stats = {}
for col in ['t_retrieval', 't_generation', 't_total']:
    vals = df_ok[col].dropna()
    stats[col] = {
        'p50':  round(np.percentile(vals, 50),  1),
        'p95':  round(np.percentile(vals, 95),  1),
        'p99':  round(np.percentile(vals, 99),  1),
        'mean': round(vals.mean(), 1),
        'std':  round(vals.std(),  1),
        'min':  round(vals.min(),  1),
        'max':  round(vals.max(),  1),
    }

# Throughput
total_time_s  = df_ok['t_total'].sum() / 1000
throughput_qpm = round(len(df_ok) / total_time_s * 60, 2)
error_rate     = round(errors / len(sample_qs) * 100, 1)

print('\n' + '='*60)
print('  INFORME DE LATENCIA — RAG iTimeControl')
print('='*60)
print(f'  Consultas totales : {len(sample_qs)}')
print(f'  Exitosas          : {len(df_ok)}')
print(f'  Tasa de error     : {error_rate}%')
print(f'  Throughput        : {throughput_qpm} queries/min')
print()
for col, label in [('t_retrieval','Retrieval (FAISS)'), ('t_generation','Generación (Claude)'), ('t_total','Total')]:
    s = stats[col]
    print(f'  {label}:')
    print(f'    p50={s["p50"]}ms  p95={s["p95"]}ms  p99={s["p99"]}ms')
    print(f'    media={s["mean"]}ms  std={s["std"]}ms  min={s["min"]}ms  max={s["max"]}ms')
    print()
print('='*60)

# Guardar stats
latency_report = {
    'n_queries': len(sample_qs),
    'n_ok':      len(df_ok),
    'error_rate_pct': error_rate,
    'throughput_qpm': throughput_qpm,
    'stats_ms':  stats,
}
with open(LOGS_DIR / 'latency_report.json', 'w', encoding='utf-8') as f:
    json.dump(latency_report, f, indent=2)
df_lat.to_csv(LOGS_DIR / 'latency_records.csv', index=False)
print('Guardado: logs/latency_report.json  |  logs/latency_records.csv')

In [ ]:
# ── Gráficas de latencia ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Informe de Latencia — RAG iTimeControl (Claude API + FAISS)', fontsize=13)

# 1. Histograma de latencia total
ax = axes[0, 0]
ax.hist(df_ok['t_total'], bins=10, color=COLORS[1], edgecolor='white', alpha=0.85)
ax.axvline(stats['t_total']['p50'], color='red',    linestyle='--', linewidth=1.5, label=f'p50={stats["t_total"]["p50"]}ms')
ax.axvline(stats['t_total']['p95'], color='orange', linestyle='--', linewidth=1.5, label=f'p95={stats["t_total"]["p95"]}ms')
ax.set_xlabel('Latencia total (ms)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de latencia total')
ax.legend(fontsize=9)

# 2. Boxplot por componente
ax = axes[0, 1]
boxdata = [df_ok['t_retrieval'].dropna().values,
           df_ok['t_generation'].dropna().values,
           df_ok['t_total'].dropna().values]
bp = ax.boxplot(boxdata, patch_artist=True, widths=0.5,
                labels=['Retrieval\n(FAISS)', 'Generación\n(Claude)', 'Total'])
for patch, color in zip(bp['boxes'], [COLORS[0], COLORS[2], COLORS[1]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Latencia (ms)')
ax.set_title('Distribución por componente')
ax.grid(True, alpha=0.4, axis='y')

# 3. Timeline de latencia por consulta
ax = axes[1, 0]
x_ids = df_ok['query_id'].values
ax.bar(x_ids, df_ok['t_retrieval'],  label='Retrieval', color=COLORS[0], alpha=0.8)
ax.bar(x_ids, df_ok['t_generation'], bottom=df_ok['t_retrieval'],
       label='Generación', color=COLORS[2], alpha=0.8)
ax.axhline(stats['t_total']['p50'], color='red',    linestyle='--', linewidth=1.2, label=f'p50={stats["t_total"]["p50"]}ms')
ax.axhline(stats['t_total']['p95'], color='orange', linestyle='--', linewidth=1.2, label=f'p95={stats["t_total"]["p95"]}ms')
ax.set_xlabel('Consulta #')
ax.set_ylabel('Latencia (ms)')
ax.set_title('Desglose de latencia por consulta')
ax.legend(fontsize=9)

# 4. Resumen en tabla visual
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['Métrica', 'Retrieval', 'Generación', 'Total'],
    ['p50 (ms)',   str(stats['t_retrieval']['p50']),   str(stats['t_generation']['p50']),   str(stats['t_total']['p50'])],
    ['p95 (ms)',   str(stats['t_retrieval']['p95']),   str(stats['t_generation']['p95']),   str(stats['t_total']['p95'])],
    ['p99 (ms)',   str(stats['t_retrieval']['p99']),   str(stats['t_generation']['p99']),   str(stats['t_total']['p99'])],
    ['Media (ms)', str(stats['t_retrieval']['mean']),  str(stats['t_generation']['mean']),  str(stats['t_total']['mean'])],
    ['', '', '', ''],
    ['Throughput', f'{throughput_qpm} q/min', '', ''],
    ['Tasa error', f'{error_rate}%',          '', ''],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.3, 1.6)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#4C72B0')
        cell.set_text_props(color='white', fontweight='bold')
ax.set_title('Resumen estadístico de latencia', fontweight='bold', pad=20)

plt.tight_layout()
out = LOGS_DIR / 'informe_latencia.png'
plt.savefig(str(out), bbox_inches='tight', dpi=130)
plt.show()
print(f'Figura guardada: {out}')

---
## 4. Optimizaciones Probadas

Se evalúa el impacto de variar el parámetro **top_k** (número de chunks recuperados)  
en la latencia y en la calidad de las respuestas.

In [ ]:
# ── Optimización top_k: carga desde CSV si ya existe ─────────────────────────
opt_csv = LOGS_DIR / "optimization_topk.csv"

if opt_csv.exists():
    df_opt = pd.read_csv(opt_csv)
    print(f"Resultados de optimización cargados desde: {opt_csv}")
    print(df_opt.to_string(index=False))
else:
    from src.evaluation.metrics import evaluate_single
    TOP_K_VALUES   = [3, 5, 7, 10]
    N_OPT_QUERIES  = 10
    opt_sample     = sample_qs[:N_OPT_QUERIES]
    optimization_results = []

    for top_k in TOP_K_VALUES:
        print(f"\nProbando top_k={top_k}...")
        times, rouge1s = [], []
        for item in opt_sample:
            try:
                t0 = time.perf_counter()
                ret = rag.retriever.search(item["question"], top_k=top_k)
                ctx = rag.retriever.format_context(ret)
                msg = f"Contexto de iTimeControl:\n{ctx}\n\nPregunta: {item['question']}" if ctx else item["question"]
                ans = rag._call_llm(msg)
                t   = (time.perf_counter() - t0) * 1000
                sc  = evaluate_single(ans, item["answer"])
                times.append(t); rouge1s.append(sc["rouge1"])
            except Exception as e:
                print(f"  Error: {e}")
        r = {
            "top_k":       top_k,
            "lat_p50_ms":  round(float(np.percentile(times, 50)), 1) if times else None,
            "lat_p95_ms":  round(float(np.percentile(times, 95)), 1) if times else None,
            "lat_mean_ms": round(float(np.mean(times)),           1) if times else None,
            "rouge1_mean": round(float(np.mean(rouge1s)),         4) if rouge1s else None,
            "n_ok":        len(times),
        }
        optimization_results.append(r)
        print(f"  p50={r['lat_p50_ms']}ms  p95={r['lat_p95_ms']}ms  ROUGE-1={r['rouge1_mean']}")

    df_opt = pd.DataFrame(optimization_results)
    df_opt.to_csv(opt_csv, index=False)
    print(f"\nGuardado: {opt_csv}")

print("\n=== RESULTADOS DE OPTIMIZACION top_k ===")
print(df_opt.to_string(index=False))

In [ ]:
# ── Gráfica de optimizaciones ─────────────────────────────────────────────────
TOP_K_VALUES = [3, 5, 7, 10]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimizacion: Impacto del top_k en Latencia y Calidad', fontsize=13)

df_opt_v = df_opt.dropna()

ax = axes[0]
ax.plot(df_opt_v['top_k'], df_opt_v['lat_p50_ms'],  'o-',  color=COLORS[0], linewidth=2, label='p50')
ax.plot(df_opt_v['top_k'], df_opt_v['lat_p95_ms'],  's--', color=COLORS[2], linewidth=2, label='p95')
ax.plot(df_opt_v['top_k'], df_opt_v['lat_mean_ms'], '^:',  color=COLORS[3], linewidth=2, label='media')
for _, row in df_opt_v.iterrows():
    ax.annotate(f'{row["lat_p50_ms"]}ms', (row['top_k'], row['lat_p50_ms']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_xlabel('top_k')
ax.set_ylabel('Latencia (ms)')
ax.set_title('Latencia vs top_k')
ax.set_xticks(TOP_K_VALUES)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)

ax2 = axes[1]
bars = ax2.bar(df_opt_v['top_k'].astype(str), df_opt_v['rouge1_mean'],
               color=COLORS[1], edgecolor='white', alpha=0.85, width=0.5)
for bar, val in zip(bars, df_opt_v['rouge1_mean']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('top_k')
ax2.set_ylabel('ROUGE-1 (media)')
ax2.set_title('Calidad de respuesta vs top_k')
ax2.set_ylim(0, max(df_opt_v['rouge1_mean']) * 1.35 if len(df_opt_v) else 1)
ax2.grid(True, alpha=0.4, axis='y')

plt.tight_layout()
out = LOGS_DIR / 'optimizacion_topk.png'
plt.savefig(str(out), bbox_inches='tight', dpi=130)
plt.show()
print(f'Figura guardada: {out}')

---
## 5. Evidencia Consolidada

In [ ]:
# ── Inventario de artefactos generados ───────────────────────────────────────
artifacts = [
    # (archivo, descripcion, notebook origen)
    ('logs/baseline_metrics.json',        'Métricas de todos los baselines (TF-IDF, NB, KNN)',    '03_baseline'),
    ('logs/baseline_resultados.png',      'Gráfica comparativa de modelos baseline',               '03_baseline'),
    ('logs/baseline_detalle_metricas.png','Métricas por consulta — Baseline TF-IDF',               '03_baseline'),
    ('logs/baseline_nb_cv.png',           'Distribución de intenciones + CV Naive Bayes',          '03_baseline'),
    ('logs/baseline_nb_vs_knn.png',       'Comparación NB vs KNN por fold',                       '03_baseline'),
    ('logs/rag_evaluation_results.json',  'Resultados detallados de evaluación RAG',               'rag_eval.py'),
    ('logs/comparativo_baseline_rag.csv', 'Tabla comparativa: baseline vs RAG',                   '08_comparativo'),
    ('logs/comparativo_tecnico.png',      'Gráfica comparativa técnica + radar',                  '08_comparativo'),
    ('logs/percepcion_usuario.png',       'Heatmap y radar de percepción del usuario',            '08_comparativo'),
    ('logs/latency_report.json',          'Informe de latencia: p50/p95/throughput/errores',       '08_comparativo'),
    ('logs/latency_records.csv',          'Registros individuales de latencia por consulta',       '08_comparativo'),
    ('logs/informe_latencia.png',         'Gráficas de distribución y timeline de latencia',      '08_comparativo'),
    ('logs/optimization_topk.csv',        'Resultados de optimización por top_k',                  '08_comparativo'),
    ('logs/optimizacion_topk.png',        'Latencia y ROUGE-1 vs top_k',                         '08_comparativo'),
    ('logs/hpo_all_trials.csv',           'Todos los trials de HPO (Random Search vs Bayesian)',  '07_hpo'),
    ('logs/hpo_decision.json',            'Configuración ganadora de HPO',                        '07_hpo'),
]

df_art = pd.DataFrame(artifacts, columns=['Archivo', 'Descripción', 'Notebook origen'])
df_art['Existe'] = df_art['Archivo'].apply(lambda p: '✅' if (ROOT / p).exists() else '❌')

print('=== INVENTARIO DE EVIDENCIA ===')
print(df_art[['Existe', 'Archivo', 'Notebook origen']].to_string(index=False))
print(f'\nTotal: {len(df_art)} artefactos | '
      f'Presentes: {(df_art["Existe"]=="✅").sum()} | '
      f'Pendientes: {(df_art["Existe"]=="❌").sum()}')

df_art.to_csv(LOGS_DIR / 'evidencia_inventario.csv', index=False)
print('Guardado: logs/evidencia_inventario.csv')

In [ ]:
# ── Tabla resumen final de todas las métricas ─────────────────────────────────
print('\n' + '='*70)
print('  RESUMEN FINAL — SISTEMA RAG iTimeControl')
print('='*70)

# Comparativo técnico
print('\n[1] MÉTRICAS DE CALIDAD DE RESPUESTA')
print(df_compare[['Métrica','TF-IDF (Baseline)','RAG (Propuesto)','Δ vs TF-IDF (%)']].to_string(index=False))

# Percepción del usuario
print('\n[2] PERCEPCIÓN DEL USUARIO (escala 1–5)')
for d in dims:
    print(f'  {d.capitalize():<14}: {df_perc[d].mean():.2f}')
print(f'  {"Score global":<14}: {df_perc["score_total"].mean():.2f}')

# Latencia
print('\n[3] LATENCIA')
print(f'  p50 total  : {stats["t_total"]["p50"]} ms')
print(f'  p95 total  : {stats["t_total"]["p95"]} ms')
print(f'  Throughput : {throughput_qpm} queries/min')
print(f'  Tasa error : {error_rate}%')

# Mejor configuración de top_k
if len(df_opt_v) > 0:
    best_topk = df_opt_v.loc[df_opt_v['rouge1_mean'].idxmax()]
    print(f'\n[4] MEJOR CONFIGURACIÓN (top_k={int(best_topk["top_k"])})')
    print(f'  ROUGE-1  : {best_topk["rouge1_mean"]}')
    print(f'  p50      : {best_topk["lat_p50_ms"]} ms')

print('\n' + '='*70)

# Guardar resumen JSON
summary = {
    'fecha': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    'comparativo_tecnico': df_compare.to_dict(orient='records'),
    'percepcion_usuario': {
        d: round(df_perc[d].mean(), 2) for d in dims + ['score_total']
    },
    'latencia': {
        'p50_ms':        stats['t_total']['p50'],
        'p95_ms':        stats['t_total']['p95'],
        'throughput_qpm': throughput_qpm,
        'error_rate_pct': error_rate,
    },
    'mejor_topk': int(best_topk['top_k']) if len(df_opt_v) > 0 else None,
}
with open(LOGS_DIR / 'resumen_semana13.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('Resumen guardado: logs/resumen_semana13.json')